# Tech Challenge - Fase 1

## Desafio

Um grande hospital universitário busca implementar um sistema inteligente de suporte ao diagnóstico, capaz de ajudar médicos e equipes clínicas na análise inicial de exames e no processamento de dados médicos.

Com um volume crescente de pacientes e exames, como radiografias, tomografias, ressonâncias e prontuários digitalizados, o hospital precisa de soluções que acelerem a triagem e apoiem as decisões médicas, reduzindo erros e otimizando o tempo dos profissionais.

Nesta primeira fase, o desafio é criar a base do sistema de IA focado em machine learning, permitindo que resultados de exames sejam analisados automaticamente e destacando informações relevantes para o diagnóstico.

## Objetivo

Construir uma solução inicial com foco em IA para processamento de exames médicos e documentos clínicos, aplicando fundamentos essenciais de IA, Machine Learning e Visão Computacional.

## Patologia escolhida para o desafio: Acidente Vascular Cerebral (AVC)

### O que é o AVC

O Acidente Vascular Cerebral (AVC) é uma condição em que o fluxo de sangue para uma parte do cérebro é interrompido. Como as células cerebrais dependem de oxigênio e nutrientes trazidos pelo sangue, essa interrupção pode causar danos neurológicos em poucos minutos.

### Tipos de AVC

1. AVC Isquêmico

- Ocorre quando um vaso sanguíneo é bloqueado por um coágulo.
- É o tipo mais comum.
- Geralmente relacionado a aterosclerose, fibrilação atrial e fatores de risco cardiovasculares.

2. AVC Hemorrágico
- Acontece quando um vaso sanguíneo se rompe, causando sangramento no cérebro.
- Pode ser causado por hipertensão não controlada, aneurismas ou malformações vasculares.

### Base de Dados - NHANES (https://www.cdc.gov/nchs/nhanes)

O NHANES (National Health and Nutrition Examination Survey) é uma base de dados pública conduzida pelo CDC (Centers for Disease Control and Prevention) dos Estados Unidos, voltada à avaliação do estado de saúde e nutricional da população norte-americana.

De forma breve:

- Natureza: estudo observacional, transversal, com amostragem probabilística representativa da população dos EUA.

- Periodicidade: realizado continuamente, organizado em ciclos bienais (ex.: 2017–2018 https://wwwn.cdc.gov/Nchs/Nhanes/continuousnhanes/default.aspx?BeginYear=2017).

- Conteúdo: combina

    - Questionários (doenças autorreferidas, estilo de vida, tabagismo, diabetes, histórico de AVC),

    - Exames físicos (pressão arterial, IMC, medidas corporais),

    - Exames laboratoriais (glicemia, hemoglobina glicada, colesterol, triglicerídeos, marcadores inflamatórios).

- Formato dos dados: arquivos modulares (.XPT), integrados por um identificador único do participante (SEQN).

- Aspecto metodológico importante: inclui pesos amostrais e variáveis de desenho complexo, permitindo inferência populacional.

No contexto acadêmico, o NHANES é amplamente utilizado para modelagem de risco cardiovascular e de AVC, epidemiologia, análise de fatores de risco e aplicações de machine learning em saúde, sendo valorizado pela qualidade dos dados, transparência metodológica e acesso aberto.

**Importando a base de dados**

In [ ]:
import pandas as pd  # importa a biblioteca pandas

pd.set_option('display.max_columns', None)  # configura o pandas para mostrar todas as colunas ao exibir DataFrames

import pandas as pd  # importa a biblioteca pandas

pd.set_option('display.max_columns', None)  # configura o pandas para mostrar todas as colunas ao exibir DataFrames

cycle_map = {
    "2005-2006": "_D",
    "2007-2008": "_E",
    "2009-2010": "_F",
    "2011-2012": "_G",
    "2013-2014": "_H",
    "2015-2016": "_I",
    "2017-2018": "_J"
}

modules = {  # dicionário que mapeia nomes lógicos para templates de arquivos XPT
    "demo": "DEMO{}.XPT",  # arquivo DEMO (dados demográficos): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.htm (ajustado para usar a letra do ciclo)
    "bpx": "BPX{}.XPT",     # arquivo BPX (medidas de pressão arterial): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BPX_J.htm
    "bpq": "BPQ{}.XPT", # arquivo BPQ (pressão arterial e colesterol): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BPQ_J.htm
    "ocq": "OCQ{}.XPT",     # arquivo OCQ (questionário de ocupação): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/OCQ_J.htm
    "ghb": "GHB{}.XPT",     # arquivo GHB (hemoglobina glicada): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/GHB_J.htm
    "bmx": "BMX{}.XPT",     # arquivo BMX (medidas corporais): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BMX_J.htm
    "smq": "SMQ{}.XPT",     # arquivo SMQ (questionário de tabagismo): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/SMQ_J.htm
    "mcq": "MCQ{}.XPT"     # arquivo de questionário médico (histórico de doenças): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/MCQ_J.htm
}

dfs = {name: [] for name in modules}  # dicionário de listas para armazenar os DataFrames de cada módulo por ciclo

cycles = list(cycle_map.keys())  # lista de ciclos (ex.: ["2011-2012", "2017-2018"])

for cycle in cycles:
    letter = cycle_map[cycle]  # obtém a letra do ciclo (ex.: "G" para "2011-2012")
    year = cycle.split('-')[0]  # obtém o ano inicial do ciclo (ex.: "2011" para "2011-2012")
    base_url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/" + year + "/DataFiles/"  # URL base para os arquivos NHANES, usando o ano inicial
    for name, file_template in modules.items():  # itera sobre (nome, template de arquivo) do dicionário modules
        file = file_template.format(letter)  # formata o template com a letra do ciclo (ex.: "DEMO_G.XPT")
        print(f"Carregando {file} para o ciclo {cycle}...")
        dfs[name].append(pd.read_sas(base_url + file))  # lê o arquivo XPT remoto e adiciona à lista de dfs[name]

# concatena os DataFrames de cada módulo de todos os ciclos
demo_df = pd.concat(dfs["demo"], ignore_index=True)
bpx_df = pd.concat(dfs["bpx"], ignore_index=True)
bpq_df = pd.concat(dfs["bpq"], ignore_index=True)
ocq_df = pd.concat(dfs["ocq"], ignore_index=True)
ghb_df = pd.concat(dfs["ghb"], ignore_index=True)
bmx_df = pd.concat(dfs["bmx"], ignore_index=True)
smq_df = pd.concat(dfs["smq"], ignore_index=True)
mcq_df = pd.concat(dfs["mcq"], ignore_index=True)

# inicia com o DataFrame 'demo' concatenado e faz merges left sucessivos com os outros DataFrames concatenados usando 'SEQN' como chave
df = demo_df \
    .merge(bpx_df, on="SEQN", how="left") \
    .merge(bpq_df, on="SEQN", how="left") \
    .merge(ocq_df, on="SEQN", how="left") \
    .merge(ghb_df, on="SEQN", how="left") \
    .merge(bmx_df, on="SEQN", how="left") \
    .merge(smq_df, on="SEQN", how="left") \
    .merge(mcq_df, on="SEQN", how="left")

df.head()  # exibe a forma (número de linhas e colunas) do DataFrame final combinado


In [ ]:
df.shape

In [ ]:
# manter somente as colunas solicitadas (se existirem) e avisar se alguma estiver ausente
cols_to_keep = [
    "SEQN", # Respondent sequence number
    "RIAGENDR", # Gender (DEMO): 1 - Male, 2 - Female
    "RIDAGEYR", # Age in years at screening (DEMO): 0 YEARS - 150 YEARS
    "BPQ020", # Ever told you had high blood pressure (BPQ): 1 - Yes, 2 - No, 7 - Refused, 9 - Don't know
    "MCQ160B",  # Ever told had congestive heart failure (MCQ): 1 - Yes, 2 - No, 7 - Refused, 9 - Don't know
    "DMDMARTL", # Marital status (DEMO): 1 - Married, 2 - Widowed, 3 - Divorced, 4 - Separated, 5 - Never married, 6 - Living with partner
    "OCQ260", # Description of job/work situation (OCQ): 1 - An employee of a private company, business, or individual for wages, salary, or commission., 2 - A federal government employee, 3 - A state government employee, 4 - A local government employee, 5 - Self-employed in own business, Working without pay in family business or farm, 6 - Refused, 77 - Don't know, 99 - Refused
    "BPXSY1", # Systolic blood pressure (BPX): 72 to 228 mm Hg
    "LBXGH", # Glycohemoglobin (GHB)
    "BMXBMI", # Body Mass Index (BMX)
    "SMQ020", # Ever smoked a cigar even 1 time? (SMQ): 1 - Yes, 2 - No, 7 - Refused, 9 - Don't know
    "MCQ160F" # TARGET -> Ever told you had a stroke (MCQ): 1 - Yes, 2 - No, 7 - Refused, 9 - Don't know
]

# verifica colunas ausentes
missing = [c for c in cols_to_keep if c not in df.columns]
if missing:
    print("Aviso: colunas ausentes e que não foram selecionadas:", missing)

# filtra o DataFrame para manter somente as colunas desejadas que existem
df = df[[c for c in cols_to_keep if c in df.columns]].copy()

# renomeia as colunas usando sufixos descritivos
suffix_descriptions = {
    "SEQN": "id", # Respondent sequence number
    "RIAGENDR": "gender", # Gender
    "RIDAGEYR": "age", # Age in years
    "BPQ020": "high_bp", # High blood pressure
    "MCQ160B": "chf", # Congestive heart failure
    "DMDMARTL": "marital", # Marital status
    "OCQ260": "occupation", # Occupation
    "BPXSY1": "sbp", # Systolic blood pressure
    "LBXGH": "hba1c", # Glycohemoglobin
    "BMXBMI": "bmi", # Body Mass Index
    "SMQ020": "smoking", # Smoking
    "MCQ160F": "stroke" # Ever told you had a stroke
}

# renomeia as colunas do DataFrame
rename_map = {c: f"{c}_{suffix_descriptions[c]}" for c in df.columns if c in suffix_descriptions}
df = df.rename(columns=rename_map)

print("Colunas renomeadas:", list(rename_map.values()))

df.shape

**Exploração de dados**

In [ ]:
df.head(5)

In [ ]:
# verificar as 20 colunas com mais valores ausentes
df.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# calcular o percentual de valores ausentes por coluna
total = len(df)

# exibir as 20 colunas com maior percentual de valores ausentes
(df.isnull().sum() / total * 100).sort_values(ascending=False).head(20)

In [ ]:
import missingno as msno

# visualizar os valores ausentes no DataFrame
msno.matrix(df)

In [ ]:
# visualizar o mapa de calor dos valores ausentes
msno.heatmap(df)

In [ ]:
# exibir a forma inicial do DataFrame
print("Initial shape:", df.shape)

# Exibir missing por coluna
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print("Missing % per column:\n", missing_pct)

# Remover todas as linhas que possuem pelo menos um valor nulo
print('Shape antes do dropna:', df.shape)

# remover linhas com valores nulos
df = df.dropna()

print('Shape após remover todas as linhas com valores nulos:', df.shape)

# Verificar valores ambíguos nas variáveis categóricas e no alvo (códigos 7 e 9)
cat_cols = ['BPQ020_high_bp', 'MCQ160B_chf', 'SMQ020_smoking', 'MCQ160F_stroke']

# exibir contagens de valores para as colunas categóricas
rows_after = df.shape[0]

# exibir contagens de valores para as colunas categóricas
for c in cat_cols:
    if c in df.columns:
        print(c, "contagens de valor:")
        print(df[c].value_counts(dropna=False))

# Remover códigos ambíguos (7,9) das colunas categóricas e do alvo (se existirem)
amb_cols = [c for c in cat_cols if c in df.columns]
rows_before = df.shape[0]
for c in amb_cols:
    df = df[~df[c].isin([7,9])]
rows_after = df.shape[0]
print(f"Linhas removidas devido a códigos ambíguos. (7/9): {rows_before - rows_after}")

df.shape

In [ ]:
df.head()

In [ ]:
# Verificar o DataFrame final
df.info()

In [ ]:
# visualizar o mapa de calor dos valores ausentes
msno.matrix(df)

In [ ]:
# verificar os valores únicos da coluna alvo (original codes)
target_col = 'MCQ160F_stroke'

if target_col in df.columns:    print("Valores alvo unicos:", sorted(df[target_col].unique()))

In [ ]:
# Manter só 1 (Yes) e 2 (No) para as features categóricas
categorical_features = ['RIAGENDR_gender','BPQ020_high_bp', 'MCQ160B_chf', 'SMQ020_smoking']

for col in categorical_features:
    if col in df.columns:
        # Manter só 1 (Yes) e 2 (No)
        df = df[df[col].isin([1, 2])].copy()
        # Criar coluna binária
        binary_col = f"{col}_bin"
        df[binary_col] = df[col].map({1: 1, 2: 0})

        # Remove as colunas originais
        df = df.drop(columns=[col])

        print(f"\n{col}:")
        print(f"Counts (1=Yes, 0=No):")
        print(df[binary_col].value_counts())


# Criar coluna binária para DMDMARTL_marital
# 1 = já foi casado (1,2,3,4,6), 0 = nunca casou (5)
df['DMDMARTL_married_bin'] = df['DMDMARTL_marital'].apply(
    lambda x: 1 if x in [1.0, 2.0, 3.0, 4.0, 6.0] else (0 if x == 5.0 else x)
)

df = df.drop('DMDMARTL_marital', axis=1)

print("\nDMDMARTL_marital -> DMDMARTL_married_bin:")
print(f"Counts (1=already married, 0=never married):")
print(df['DMDMARTL_married_bin'].value_counts())

df.head()

In [ ]:
# Manter só 1 (Yes) e 2 (No) no alvo e criar coluna binária 'MCQ160F_stroke_bin'
target_col = 'MCQ160F_stroke'
if target_col in df.columns:
    df = df[df[target_col].isin([1,2])].copy()
    df['MCQ160F_stroke_bin'] = df[target_col].map({1:1, 2:0})
    print("Counts (1=stroke,0=no):")
    print(df['MCQ160F_stroke_bin'].value_counts())
    
    # Remove a coluna original
    df = df.drop(columns=[target_col])


In [ ]:
# Resumo estatístico das colunas numéricas
df.describe()

**Analisando as variáveis numéricas**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sb

# 1) Heatmap de correlação (numéricas)
num_cols = ['RIAGENDR_gender_bin', 'RIDAGEYR_age', 'BPQ020_high_bp_bin', 'MCQ160B_chf_bin', 'DMDMARTL_married_bin', 'OCQ260_occupation', 'SMQ020_smoking_bin', 'MCQ160F_stroke_bin']
# Keep only numeric columns for correlation (convert categories to numeric if needed)
corr_df = df[num_cols].copy()
corr_df = corr_df.apply(pd.to_numeric, errors='coerce')
plt.figure(figsize=(8,6))
sb.heatmap(corr_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlação entre variáveis numéricas')
plt.tight_layout()

plt.show()


In [ ]:
# Taxa de AVC por faixa etária
df['age_bin'] = pd.cut(df['RIDAGEYR_age'], bins=range(0, 101, 10))
age_rate = df.groupby('age_bin')['MCQ160F_stroke_bin'].mean().reset_index()
plt.figure(figsize=(8,4))
sb.barplot(x='age_bin', y='MCQ160F_stroke_bin', data=age_rate, palette='viridis')
plt.xticks(rotation=45)
plt.ylabel('Taxa de AVC')
plt.title('Taxa de AVC por faixa etária (bins de 10 anos)')
plt.tight_layout()
plt.show()

In [ ]:
# Taxa de AVC por variáveis categóricas (BPQ020_high_bp, MCQ160B_chf, SMQ020_smoking)
label_map = {1: 'Sim', 2: 'Não'}
for var in ['BPQ020_high_bp', 'MCQ160B_chf', 'SMQ020_smoking']:
    if var in df.columns:
        agg = df.groupby(var)['MCQ160F_stroke_bin'].agg(['count','sum']).reset_index()
        agg['rate'] = agg['sum'] / agg['count']
        plt.figure(figsize=(5,3))
        sb.barplot(x=var, y='rate', data=agg, palette='coolwarm')
        plt.xticks(ticks=range(len(agg[var])), labels=[label_map.get(int(v), v) for v in agg[var]])
        plt.ylim(0, agg['rate'].max()*1.15)
        plt.ylabel('Taxa de AVC')
        plt.title(f'Taxa de AVC por {var}')
        plt.tight_layout()
        plt.show()

In [ ]:
# Distribuição das variáveis numéricas (selecionadas)
num_hist_cols = ['RIDAGEYR_age', 'BPXSY1_sbp', 'LBXGH_hba1c', 'BMXBMI_bmi']
df[num_hist_cols].hist(bins=50, figsize=(20,15))

# Ajustar layout e mostrar os histogramas
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc, confusion_matrix

num_cols = ['RIDAGEYR_age', 'BPXSY1_sbp', 'LBXGH_hba1c', 'BMXBMI_bmi']
cat_cols = ['RIAGENDR_gender_bin', 'DMDMARTL_married_bin', 'BPQ020_high_bp_bin', 'MCQ160B_chf_bin', 'SMQ020_smoking_bin']

# Eliminar linhas com alvo ausente
df_model = df.dropna(subset=['MCQ160F_stroke_bin']).copy()
X = df_model[num_cols + cat_cols]
y = df_model['MCQ160F_stroke_bin']

# Dividir em treino e teste com estratificação
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Pipeline de pré-processamento
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))])

# Combinar pipelines numéricos e categóricos
preproc = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])

pipe_lr = Pipeline([('pre', preproc), ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))])
pipe_rf = Pipeline([('pre', preproc), ('clf', RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42))])

# Treinar os modelos
pipe_lr.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)

# Avaliar os modelos no conjunto de teste
for name, model in [('LogisticRegression', pipe_lr), ('RandomForest', pipe_rf)]:
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    print(f"===== {name} =====")
    print(classification_report(y_test, y_pred))
    print('ROC AUC:', roc_auc_score(y_test, y_prob))
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    print('PR AUC:', auc(recall, precision))
    print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))

# Importância da permutação (alternativa ao SHAP)
from sklearn.inspection import permutation_importance

# Calcular a importância da permutação nas características transformadas (após o pré-processamento).
X_test_trans = preproc.transform(X_test)
r = permutation_importance(pipe_rf.named_steps['clf'], X_test_trans, y_test, n_repeats=30, random_state=42, scoring='roc_auc')
feat_names = preproc.get_feature_names_out()

# Criar DataFrame de importâncias
imp_df = pd.DataFrame({'feature': feat_names, 'importance': r.importances_mean, 'std': r.importances_std})
imp_df = imp_df.sort_values('importance', ascending=False)

print('\nAs 10 principais características por importância de permutação (RandomForest, características transformadas):')
print(imp_df.head(10))

In [ ]:
# Rebalanceamento por undersampling da classe majoritária
from sklearn.utils import resample

# Separar as classes
df_major = df_model[df_model['MCQ160F_stroke_bin'] == 0]
df_minor = df_model[df_model['MCQ160F_stroke_bin'] == 1]

# Reduzir classe majoritária para o tamanho da minoritária
df_major_down = resample(df_major, 
                         replace=True, 
                         n_samples=len(df_minor), 
                         random_state=42)

# Concatenar e embaralhar
df_balanced = pd.concat([df_major_down, df_minor]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Distribuição após undersampling:")
print(df_balanced['MCQ160F_stroke_bin'].value_counts())

# Redefinir X e y
X_bal = df_balanced[num_cols + cat_cols]
y_bal = df_balanced['MCQ160F_stroke_bin']

# Dividir em treino e teste
X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(X_bal, y_bal, stratify=y_bal, test_size=0.2, random_state=42)

# Treinar e avaliar ambos os modelos com base balanceada
for name, model in [('LogisticRegression', pipe_lr), ('RandomForest', pipe_rf)]:
    model.fit(X_train_bal, y_train_bal)
    y_pred_bal = model.predict(X_test_bal)
    y_prob_bal = model.predict_proba(X_test_bal)[:,1]
    
    print(f"===== {name} (base balanceada) =====")
    print(classification_report(y_test_bal, y_pred_bal))
    print('ROC AUC:', roc_auc_score(y_test_bal, y_prob_bal))
    precision, recall, _ = precision_recall_curve(y_test_bal, y_prob_bal)
    print('PR AUC:', auc(recall, precision))
    print('Confusion matrix:\n', confusion_matrix(y_test_bal, y_pred_bal))
    print()

In [ ]:
# Plote as principais permutações de importância (selecione as 20 principais para melhor visualização).
top_k = min(20, len(imp_df))

# Ordene o gráfico de barras horizontais em ordem crescente (com os elementos mais importantes no topo).
top = imp_df.head(top_k).sort_values('importance', ascending=True)
plt.figure(figsize=(8, max(4, 0.35*top_k)))
plt.barh(top['feature'], top['importance'], xerr=top['std'], color='C0')
plt.xlabel('Importância')
plt.title('Importância da permutação (RandomForest, AUC ROC)')
plt.tight_layout()
plt.show()

In [ ]:
import pickle

# Salvar o modelo RandomForest treinado com base balanceada
with open('pipe_rf_model.pkl', 'wb') as f:
    pickle.dump(pipe_rf, f)

# Escolha dois registros aleatorios da base de teste balanceada (um com AVC e outro sem AVC) para teste do modelo salvo
test_samples = pd.concat([
    X_test_bal[y_test_bal == 1].sample(n=1, random_state=42),
    X_test_bal[y_test_bal == 0].sample(n=1, random_state=42)
])

# Adicionar coluna indicando se é stroke ou não
test_samples['MCQ160F_stroke_bin'] = y_test_bal.loc[test_samples.index]
test_samples['stroke_label'] = test_samples['MCQ160F_stroke_bin'].map({1: 'Stroke (Sim)', 0: 'No Stroke (Não)'})

test_samples.head(2)

## Referências

- https://pandascouple.medium.com/projeto-machine-learning-previs%C3%A3o-de-avc-f4b7dce11929
- https://jornal.usp.br/radio-usp/uso-de-ia-e-analise-de-dados-na-prevencao-de-avc-e-ataque-isquemico-transitorio/
- https://www.nature.com/articles/s41598-024-61665-4?error=cookies_not_supported&code=1d85f26e-6ade-4ec5-9132-5511aa615597&_x_tr_sl=en&_x_tr_tl=pt&_x_tr_hl=pt&_x_tr_pto=tc
- https://jhi.sbis.org.br/index.php/jhi-sbis/article/view/980
- https://www.nature.com/articles/s41598-025-01855-w
